In [1]:
# !pip install neo4j
from neo4j import GraphDatabase

In [2]:
# Configura i dettagli di connessione
uri = "bolt://localhost:7687"
username = "neo4j"
password = "neo4jneo4j"

In [3]:
# Inizializza il driver
driver = GraphDatabase.driver(uri,auth=(username,password))

In [4]:
# Funzione per eseguire query Cypher
def run_query(query):
    with driver.session() as session:
        result = session.run(query)
        return result.data()

In [5]:
# Esempio di query: mostra i primi 5 film
query = """
MATCH (m:Movie)
RETURN m.title, m.released
LIMIT 5
"""
results = run_query(query)
for record in results:
    print(f"Title: {record['m.title']}, Released: {record['m.released']}")

Title: The Matrix Reloaded, Released: 2003
Title: The Matrix Revolutions, Released: 2003
Title: The Devil's Advocate, Released: 1997
Title: A Few Good Men, Released: 1992
Title: Top Gun, Released: 1986


In [6]:
# Chiudi la connessione
driver.close()

In [7]:
driver = GraphDatabase.driver(uri,auth=(username,password))
# Trovare gli attori che hanno recitato con Keanu Reeves
query2 = """
MATCH (p1:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(p2:Person)
WHERE p1.name = 'Keanu Reeves' AND p2<>p1
RETURN p2.name AS co_actor, m.title AS movie
LIMIT 10
"""
results = run_query(query2)
print("Attori che hanno lavorato con Keanu Reeves\n")
for record in results:
    print(f"{record['co_actor']} in {record['movie']}")

driver.close()

Attori che hanno lavorato con Keanu Reeves

Carrie-Anne Moss in The Matrix Reloaded
Laurence Fishburne in The Matrix Reloaded
Hugo Weaving in The Matrix Reloaded
Carrie-Anne Moss in The Matrix Revolutions
Laurence Fishburne in The Matrix Revolutions
Hugo Weaving in The Matrix Revolutions
Charlize Theron in The Devil's Advocate
Al Pacino in The Devil's Advocate
Brooke Langton in The Replacements
Gene Hackman in The Replacements


In [8]:
import pandas as pd

driver = GraphDatabase.driver(uri,auth=(username,password))
# Attori che hanno partecipato a più film
query3 = """
MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
RETURN p.name AS actor, COUNT(m) AS movies_count
ORDER BY movies_count DESC
LIMIT 10
"""
results = run_query(query3)

# Converti i risultati in un DataFrame
df = pd.DataFrame(results)
print(df)

# Visualizza una statistica
print("\nAttore con più film:")
print(df.iloc[0])

driver.close()

                actor  movies_count
0           Tom Hanks            12
1        Keanu Reeves             7
2        Hugo Weaving             5
3      Jack Nicholson             5
4            Meg Ryan             5
5    Cuba Gooding Jr.             4
6          Tom Cruise             3
7         Kevin Bacon             3
8    Carrie-Anne Moss             3
9  Laurence Fishburne             3

Attore con più film:
actor           Tom Hanks
movies_count           12
Name: 0, dtype: object


In [9]:
def create_node(tx, label, properties):
    """Funzione per creare un nodo con etichetta e proprietà specifiche."""
    query = f"CREATE (n:{label} {{"
    query += ", ".join([f"{k}: ${k}" for k in properties.keys()])
    query += "}) RETURN n"
    tx.run(query, **properties)

In [10]:
def create_relationship(tx, node1_label, node1_props, rel_type, node2_label, node2_props):
    """
    Crea una relazione tra due nodi, identificati da insiemi arbitrari di proprietà.

    Args:
        tx: oggetto transaction Neo4j
        node1_label (str): etichetta del primo nodo (es. 'Person')
        node1_props (dict): proprietà per individuare il primo nodo (es. {'name': "Uma Thurman"})
        rel_type (str): tipo di relazione (es. 'ACTED_IN')
        node2_label (str): etichetta del secondo nodo
        node2_props (dict): proprietà per individuare il secondo nodo
    """
    # Costruisci i pattern di filtro per entrambi i nodi
    node1_match = ", ".join([f"{k}: ${'n1_' + k}" for k in node1_props])
    node2_match = ", ".join([f"{k}: ${'n2_' + k}" for k in node2_props])

    query = (
        f"MATCH (a:{node1_label} {{{node1_match}}}), "
        f"(b:{node2_label} {{{node2_match}}}) "
        f"CREATE (a)-[r:{rel_type}]->(b) "
        f"RETURN a, r, b"
    )

    # Combina i parametri in un solo dizionario
    params = {f"n1_{k}": v for k, v in node1_props.items()}
    params.update({f"n2_{k}": v for k, v in node2_props.items()})

    tx.run(query, **params)

In [11]:
def populate_database():
    """Popola il database con nodi e relazioni di esempio usando proprietà generiche."""
    with driver.session() as session:
        # Creazione dei nodi
        session.execute_write(create_node, "Person", {"name": "Uma Thurman", "born": 1970})
        session.execute_write(create_node, "Movie", {"title": "Pulp Fiction", "released": 1994})

        # Creazione della relazione usando la funzione generalizzata
        session.execute_write(
            create_relationship,
            "Person", {"name": "Uma Thurman"},
            "ACTED_IN",
            "Movie", {"title": "Pulp Fiction"}
        )

        print("Database popolato con successo!")

In [12]:
if __name__ == "__main__":
    driver = GraphDatabase.driver(uri,auth=(username,password))
    populate_database()
    driver.close()

Database popolato con successo!


In [13]:
def delete_elements(
    tx,
    node1_label=None,
    node1_props=None,
    rel_type=None,
    node2_label=None,
    node2_props=None,
    detach=False
):
    """
    Cancella nodi, relazioni o entrambi, in base ai parametri forniti.

    Args:
        tx: transaction Neo4j
        node1_label (str): etichetta del primo nodo (es. 'Person') oppure None
        node1_props (dict): proprietà del primo nodo (es. {'name': 'Uma Thurman'}) oppure None
        rel_type (str): tipo di relazione (es. 'ACTED_IN') oppure None
        node2_label (str): etichetta del secondo nodo (es. 'Movie') oppure None
        node2_props (dict): proprietà del secondo nodo oppure None
        detach (bool): se True, usa DETACH DELETE per rimuovere anche le relazioni collegate
    """

    match_parts = []
    params = {}

    if node1_label:
        node1_filter = ", ".join([f"{k}: ${'n1_' + k}" for k in (node1_props or {})])
        match_parts.append(f"(a:{node1_label} {{{node1_filter}}})" if node1_props else f"(a:{node1_label})")
        params.update({f"n1_{k}": v for k, v in (node1_props or {}).items()})

    if node2_label:
        node2_filter = ", ".join([f"{k}: ${'n2_' + k}" for k in (node2_props or {})])
        match_parts.append(f"(b:{node2_label} {{{node2_filter}}})" if node2_props else f"(b:{node2_label})")
        params.update({f"n2_{k}": v for k, v in (node2_props or {}).items()})

    # Costruzione della query dinamica
    if rel_type and len(match_parts) == 2:
        query = f"MATCH {match_parts[0]}-[r:{rel_type}]->{match_parts[1]} DELETE r"
    elif rel_type and len(match_parts) == 1:
        query = f"MATCH {match_parts[0]}-[r:{rel_type}]-() DELETE r"
    elif not rel_type and node1_label:
        delete_kw = "DETACH DELETE" if detach else "DELETE"
        query = f"MATCH {match_parts[0]} {delete_kw} a"
    elif not rel_type and node2_label:
        delete_kw = "DETACH DELETE" if detach else "DELETE"
        query = f"MATCH {match_parts[1]} {delete_kw} b"
    else:
        raise ValueError("Devi specificare almeno un nodo o una relazione da cancellare.")

    tx.run(query, **params)

In [14]:
def delete_all_nodes_and_relationships():
    """Funzione per cancellare tutti i nodi e relazioni nel database."""
    with driver.session() as session:
        session.execute_write(lambda tx: tx.run("MATCH(n) DETACH DELETE n"))
        print("Tutti i nodi e le relazioni sono stati cancellati.")

In [15]:
# Cancellare una relazione specifica
driver = GraphDatabase.driver(uri,auth=(username,password))
driver.session().execute_write(
    delete_elements,
    "Person", {"name": "Uma Thurman"},
    "ACTED_IN",
    "Movie", {"title": "Pulp Fiction"}
)
driver.close()

In [16]:
# Cancella tutti i nodi di tipo Person, ma lascia intatte le relazioni collegate 
# (errore se ci sono relazioni ancora presenti)
driver = GraphDatabase.driver(uri,auth=(username,password))
driver.session().execute_write(delete_elements, node1_label="Person")
driver.close()

ConstraintError: {code: Neo.ClientError.Schema.ConstraintValidationFailed} {message: Cannot delete node<0>, because it still has relationships. To delete this node, you must first delete its relationships.}

In [17]:
# Cancella tutti i nodi di tipo Person con tutte le relazioni 
# (grazie a DETACH DELETE)
driver = GraphDatabase.driver(uri,auth=(username,password))
driver.session().execute_write(delete_elements, node1_label="Person", detach=True)
driver.close()

In [18]:
# Cancella tutto un nodo specifico e le sue connessioni
driver = GraphDatabase.driver(uri,auth=(username,password))
driver.session().execute_write(delete_elements, node1_label="Movie", node1_props={"title": "Pulp Fiction"}, detach=True)
driver.close()

In [19]:
def update_node(tx, label, match_props, new_properties):
    """
    Aggiorna le proprietà di un nodo identificato da un insieme di proprietà arbitrarie.

    Args:
        tx: oggetto transaction Neo4j
        label (str): etichetta del nodo (es. 'Person')
        match_props (dict): proprietà per identificare il nodo (es. {'name': 'Uma Thurman'})
        new_properties (dict): nuove proprietà da impostare (es. {'born': 1971, 'nationality': 'USA'})
    """

    # Costruzione del filtro MATCH
    match_str = ", ".join([f"{k}: ${'m_' + k}" for k in match_props])

    # Costruzione della parte SET
    set_str = ", ".join([f"n.{k} = ${'u_' + k}" for k in new_properties])

    query = (
        f"MATCH (n:{label} {{{match_str}}}) "
        f"SET {set_str} "
        f"RETURN n"
    )

    # Parametri combinati (m_ = match, u_ = update)
    params = {f"m_{k}": v for k, v in match_props.items()}
    params.update({f"u_{k}": v for k, v in new_properties.items()})

    result = tx.run(query, **params)
    return result.single()

In [20]:
driver = GraphDatabase.driver(uri,auth=(username,password))
driver.session().execute_write(
    update_node,
    "Person",
    {"name": "Uma Thurman"},
    {"born": 1971, "nationality": "USA"}
)
driver.close()

In [21]:
def update_relationship(tx, node1_label, node1_props, rel_type, node2_label, node2_props, new_properties):
    """
    Aggiorna le proprietà di una relazione esistente tra due nodi identificati da etichette e proprietà arbitrarie.

    Args:
        tx: oggetto transaction Neo4j
        node1_label (str): etichetta del primo nodo (es. 'Person')
        node1_props (dict): proprietà per individuare il primo nodo (es. {'name': 'Uma Thurman'})
        rel_type (str): tipo di relazione (es. 'ACTED_IN')
        node2_label (str): etichetta del secondo nodo (es. 'Movie')
        node2_props (dict): proprietà per individuare il secondo nodo (es. {'title': 'Pulp Fiction'})
        new_properties (dict): proprietà da aggiornare sulla relazione (es. {'role': 'Mia Wallace'})
    """

    # MATCH dinamico per i due nodi
    node1_match = ", ".join([f"{k}: ${'n1_' + k}" for k in node1_props])
    node2_match = ", ".join([f"{k}: ${'n2_' + k}" for k in node2_props])

    # SET dinamico per le proprietà della relazione
    set_str = ", ".join([f"r.{k} = ${'u_' + k}" for k in new_properties])

    query = (
        f"MATCH (a:{node1_label} {{{node1_match}}})-[r:{rel_type}]->(b:{node2_label} {{{node2_match}}}) "
        f"SET {set_str} "
        f"RETURN r"
    )

    # Costruisci il dizionario dei parametri
    params = {f"n1_{k}": v for k, v in node1_props.items()}
    params.update({f"n2_{k}": v for k, v in node2_props.items()})
    params.update({f"u_{k}": v for k, v in new_properties.items()})

    result = tx.run(query, **params)
    return result.single()

In [22]:
driver = GraphDatabase.driver(uri,auth=(username,password))
driver.session().execute_write(
    update_relationship,
    "Person", {"name": "Uma Thurman"},
    "ACTED_IN",
    "Movie", {"title": "Pulp Fiction"},
    {"role": "Mia Wallace", "year": 1994}
)
driver.close()